<a href="https://colab.research.google.com/github/Roopanshi-Marwaha/Fraud_Ring_Detection/blob/main/Fraud_Ring_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

accounts = pd.read_csv('accounts.csv')
transactions = pd.read_csv('transactions.csv')
alerts = pd.read_csv('alerts.csv')

print(accounts.shape, transactions.shape, alerts.shape)
#printed rows and columns-->shape

(10000, 7) (1323234, 8) (1719, 9)


#EDA

ACCOUNTS

In [ ]:
accounts.head()
#to see first 5 rows
#TX_BEHAVIOR_ID--> a group number controlling how that account normally transacts (used to simulate realistic behavior)
# IS_FRAUD here means the account itself is a known fraud account, not just one transaction

,ACCOUNT_ID,CUSTOMER_ID,INIT_BALANCE,COUNTRY,ACCOUNT_TYPE,IS_FRAUD,TX_BEHAVIOR_ID
0,0,C_0,184.44,US,I,False,1
1,1,C_1,175.80,US,I,False,1
2,2,C_2,142.06,US,I,False,1
3,3,C_3,125.89,US,I,False,1
4,4,C_4,151.13,US,I,False,1


In [ ]:
accounts['ACCOUNT_TYPE'].value_counts()
#ACCOUNT_TYPE: all 10,000 accounts are "I" so there's only one account type in this dataset, not a mix.
# That column won't be useful for us later (no variation = no signal)

,count
ACCOUNT_TYPE,
I,10000


In [ ]:
accounts['IS_FRAUD'].value_counts()
# IS_FRAUD: 1,685 out of 10,000 accounts (~16.8%) are fraud accounts. That's a high fraud rate so plenty of fraud examples to learn from, not a rare event problem

,count
IS_FRAUD,
False,8315
True,1685


In [ ]:
accounts['COUNTRY'].value_counts()
#all US so this column is also useless, no variation.

,count
COUNTRY,
US,10000


In [ ]:
accounts['TX_BEHAVIOR_ID'].value_counts()
#exactly 5 behavior groups, 2,000 accounts each, perfectly even split

,count
TX_BEHAVIOR_ID,
1,2000
2,2000
3,2000
4,2000
5,2000


In [ ]:
pd.crosstab(accounts['TX_BEHAVIOR_ID'], accounts['IS_FRAUD'])
#This will show fraud count per behavior group

#fraud rates:-
# Group 1: 291/2000 = 14.6%
# Group 2: 330/2000 = 16.5%
# Group 3: 309/2000 = 15.5%
# Group 4: 358/2000 = 17.9%
# Group 5: 397/2000 = 19.9%
# There's a mild upward trend (group 5 has more fraud than group 1) but it's not a huge gap.

IS_FRAUD,False,True
TX_BEHAVIOR_ID,,
1,1709,291
2,1670,330
3,1691,309
4,1642,358
5,1603,397


So in accounts.csv. We now know:

10k accounts are there

only useful columns are IS_FRAUD (target-ish) and TX_BEHAVIOR_ID (weak feature),

COUNTRY & ACCOUNT_TYPE are dead weight.



TRANSACTIONS

In [ ]:
transactions.info()
transactions.head()

#things cleared:-
# ALERT_ID = -1 matlab is transaction ka koi alert nahi (normal transaction)
# TIMESTAMP numbers mein hai (0, 1, 2...) — real dates nahi, simulation ke "time steps" hain

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1323234 entries, 0 to 1323233
Data columns (total 8 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   TX_ID                1323234 non-null  int64  
 1   SENDER_ACCOUNT_ID    1323234 non-null  int64  
 2   RECEIVER_ACCOUNT_ID  1323234 non-null  int64  
 3   TX_TYPE              1323234 non-null  object 
 4   TX_AMOUNT            1323234 non-null  float64
 5   TIMESTAMP            1323234 non-null  int64  
 6   IS_FRAUD             1323234 non-null  bool   
 7   ALERT_ID             1323234 non-null  int64  
dtypes: bool(1), float64(1), int64(5), object(1)
memory usage: 71.9+ MB


,TX_ID,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_TYPE,TX_AMOUNT,TIMESTAMP,IS_FRAUD,ALERT_ID
0,1,6456,9069,TRANSFER,465.05,0,False,-1
1,2,7516,9543,TRANSFER,564.64,0,False,-1
2,3,2445,9356,TRANSFER,598.94,0,False,-1
3,4,2576,4617,TRANSFER,466.07,0,False,-1
4,5,3524,1773,TRANSFER,405.63,0,False,-1


In [ ]:
transactions['TX_TYPE'].value_counts()
#TX_TYPE — sab kuch "TRANSFER" hi hai, koi variation nahi. Yeh column bhi useless nikla hai

,count
TX_TYPE,
TRANSFER,1323234


In [ ]:
transactions['IS_FRAUD'].value_counts()
#IS_FRAUD — sirf 1,719 fraud transactions out of 13,23,234 = 0.13% -->bahut zyada imbalanced data
# so agar directly model train karenge toh woh "sab kuch not fraud hai" bol ke bhi 99.87% accuracy de dega but will be useless.

#now observation:-
# Account level pe fraud rate tha 16.8% (accounts.csv)
# Transaction level pe fraud rate hai sirf 0.13% (transactions.csv)
#so this means:-
# ek fraud account bhi apni zyadatar transactions normal karta hai, sirf kuch hi transactions actually "fraud-flagged" hoti hain us account ki.
# Yeh real duniya jaisa hi hai, aise account bhi mostly normal looking transfers karte hai, sirf kabhi kabhi suspicious wala move hota hai.

,count
IS_FRAUD,
False,1321515
True,1719


In [ ]:
print(transactions['TIMESTAMP'].min())
print(transactions['TIMESTAMP'].max())
print(transactions['TIMESTAMP'].nunique())

#Toh simulation mein 200 discrete time units the, aur 1.3M transactions un 200 steps mein spread hain.
#kitni jaldi jaldi paisa move ho raha hai--> yeh voh bata sakta hai

0
199
200


In [ ]:
transactions['TX_AMOUNT'].describe()
#Max = ₹2.14 crore — ek bahut hi bada outlier
# mean median se bohot zyada hai iska matlab hai distribution is right skewed.
#yani zyadatar transactions chhoti hain, lekin kuch bahut hi bade outliers hain jo average ko upar kheech rahe hain.

,TX_AMOUNT
count,1.323234e+06
mean,1.159882e+05
std,1.320091e+06
min,0.000000e+00
25%,2.393000e+01
50%,1.567100e+02
75%,4.400000e+02
max,2.147484e+07


In [ ]:
transactions.groupby('IS_FRAUD')['TX_AMOUNT'].describe()

#Fraud transactions: amount sirf ₹2.54 se ₹19.92 tak --> bohot chhote amounts hain (max ₹20!)
#Non fraud transactions: amount ₹0 se ₹2.14 crore tak --> bahut bada range

#Matlab very intresting pattern seen:-
#jitna socha tha uska ulta hai
# log sochte hain fraud = bada amount,
# lekin yahan fraud transactions chhote chhote hain.
# Yeh actually real world money laundering pattern se match karta hai: "structuring" ya "smurfing"
# i.e bade amount ko jaanbujh ke chhote chhote pieces mein tod dena taaki bank ke threshold based alerts (jo bade amount pe trigger hote hain) trigger na ho.

,count,mean,std,min,25%,50%,75%,max
IS_FRAUD,,,,,,,,
False,1321515.0,116139.044522,1.320942e+06,0.00,24.29,156.97,440.05,21474836.47
True,1719.0,9.763310,5.928078e+00,2.54,3.78,10.60,15.30,19.92


ALERTS

In [ ]:
alerts['ALERT_TYPE'].value_counts()

# cycle (936 cases): A→B→C→A. Paisa ek chain mein ghoom ke wapas apne origin ke paas ya kisi related account tak aata hai.
# fan_in (783 cases): bahut saare alag accounts se ek single account mein paisa aana.
# example, 20 chhote mule accounts, sab ek hi "collector" account ko chhoti chhoti amounts bhejte hain,
# jo baad mein woh sara paisa ek saath nikaal leta hai ya aage forward karta hai.

,count
ALERT_TYPE,
cycle,936
fan_in,783


In [ ]:
alerts.head()

# ALERT_ID = 377 do baar aaya hai (row 1 aur row 3), same TX_AMOUNT (10.27) ke saath lekin alag TX_ID, alag sender/receiver, alag timestamp.
# Iska matlab: ek ALERT_ID = ek poora fraud ring, aur uss ring ke multiple transactions hote hain jo saath mein table mein listed hain. Matlab agar cycle hai A→B→C→A, toh teeno transactions (A→B, B→C, C→A) same ALERT_ID share karenge, alag alag rows mein

,ALERT_ID,ALERT_TYPE,IS_FRAUD,TX_ID,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_TYPE,TX_AMOUNT,TIMESTAMP
0,193,fan_in,True,82,6976,9739,TRANSFER,4.85,0
1,377,cycle,True,949,5776,2570,TRANSFER,10.27,0
2,189,fan_in,True,6280,9999,9530,TRANSFER,2.74,1
3,377,cycle,True,7999,1089,7352,TRANSFER,10.27,1
4,130,fan_in,True,12975,7025,9708,TRANSFER,3.53,2


In [ ]:
print(alerts['ALERT_ID'].nunique())
print(len(alerts))
print(alerts.groupby('ALERT_ID').size().describe())


# 391 unique fraud rings total (cycle + fan_in dono milake)
# Har ring mein average ~4.4 transactions hote hain
# Sabse chhota ring = 1 transaction (WEIRD AS "ring" mein toh kam se kam 2-3 transactions honi chahiye, single transaction wala "ring" nahi ho sakta typically)--->OUTLIER
# Sabse bada ring = 5 transactions, aur woh bhi bahut consistent hai (75th percentile bhi 5 hai) so matlab zyadatar rings ka size 4 ya 5 hi hai, bahut tight range, Yeh bahut structured/simulated pattern hai real life mei not that consistent but stimulation ke liye works

391
1719
count    391.000000
mean       4.396419
std        0.678526
min        1.000000
25%        4.000000
50%        4.000000
75%        5.000000
max        5.000000
dtype: float64


In [ ]:
ring_sizes = alerts.groupby('ALERT_ID').size()
single_tx_rings = ring_sizes[ring_sizes == 1]
print(single_tx_rings)

ALERT_ID
44     1
231    1
272    1
dtype: int64


In [ ]:
alerts[alerts['ALERT_ID'].isin([44, 231, 272])]

# Ye 3 single-tx rings (231, 272, 44) sab timestamp 198/199 pe hain
# simulation ke last steps. Ring poora complete hone se pehle hi simulation khatam ho gayi (200 steps ka limit), isliye baaki  transactions record nahi hue.
# Data error nahi, bas edge effect hai jo kisi bhi time bounded simulation mein ho sakta hai (jaise ek movie ka last scene achanak kat jaye).
# Feature engineering mein to remember: kuch rings incomplete honge sirf time cutoff ki wajah se.

,ALERT_ID,ALERT_TYPE,IS_FRAUD,TX_ID,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_TYPE,TX_AMOUNT,TIMESTAMP
1704,231,cycle,True,1312016,3040,1565,TRANSFER,12.14,198
1714,272,cycle,True,1316271,2465,707,TRANSFER,16.31,198
1715,44,fan_in,True,1316636,1453,8709,TRANSFER,2.81,199


In [ ]:
alerts[alerts['ALERT_ID'] == 193]
# Receiver same (9739) teeno row mein
# 3 alag senders (6976,4596,1950)
# Amount same (4.85) har baar
# Timestamps thode alag-alag (0,4,6)

,ALERT_ID,ALERT_TYPE,IS_FRAUD,TX_ID,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_TYPE,TX_AMOUNT,TIMESTAMP
0,193,fan_in,True,82,6976,9739,TRANSFER,4.85,0
12,193,fan_in,True,24024,4596,9739,TRANSFER,4.85,4
36,193,fan_in,True,40614,1950,9739,TRANSFER,4.85,6


In [ ]:
alerts[alerts['ALERT_ID'] == 189]

# Receiver same (9530)
# Amount same (2.74) chaaron transactions mein
# 4 alag senders
#SO, fan_in rings mein receiver same + amount same hota hai, timestamps thode gap ke saath

,ALERT_ID,ALERT_TYPE,IS_FRAUD,TX_ID,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_TYPE,TX_AMOUNT,TIMESTAMP
2,189,fan_in,True,6280,9999,9530,TRANSFER,2.74,1
6,189,fan_in,True,19832,3033,9530,TRANSFER,2.74,3
21,189,fan_in,True,34189,4769,9530,TRANSFER,2.74,5
41,189,fan_in,True,44935,7703,9530,TRANSFER,2.74,7


In [ ]:
print((transactions['ALERT_ID'] != -1).sum())
print(transactions['ALERT_ID'].nunique())

#checking transactions.csv ka ALERT_ID column theek se alerts.csv se match kar raha hai
# 1,719 transactions ke paas actual alert hai (yeh transactions.csv ka fraud count bhi 1,719 tha perfectly match)

1719
392


In [ ]:
# transactions.csv se sirf woh rows nikalo jinka ALERT_ID -1 nahi hai (matlab fraud/alerted transactions)
fraud_tx = transactions[transactions['ALERT_ID'] != -1]
#ALERT_ID = -1 ka matlab hai: is transaction pe koi alert nahi laga, yeh ek normal/non fraud transaction hai.

print(fraud_tx.shape)   # kitni rows aur columns hain fraud_tx mein
print(alerts.shape)     # compare karne ke liye alerts.csv ka shape bhi dekho

#Row count match ho gaya 1,719 rows dono mein.
# Confirm: alerts.csv transactions.csv ke fraud rows ka hi ek collection hai, koi extra/missing case nahi.

(1719, 8)
(1719, 9)


In [ ]:
# dono files ke column names dekhr, common columns identify karne ke liye
print(fraud_tx.columns.tolist())
print(alerts.columns.tolist())

#Sirf ek extra column alerts.csv mein hai jo transactions.csv mein nahi hai:
# ALERT_TYPE (cycle/fan_in).


# Toh conclusion yeh hai:
#  alerts.csv = transactions.csv ke fraud rows ka exact duplicate copy, bas ek extra column (ALERT_TYPE) hai.
# Matlab hume alag se in dono ko merge karne ki zarurat nahi hai hum seedha transactions.csv (poora dataset, saari 1.3M rows) le sakte hain,
# aur usme alerts.csv se sirf ALERT_TYPE column ko ALERT_ID ke basis pe map/join kar sakte hain, taaki har fraud transaction ko pata ho woh "cycle" hai ya "fan_in".

['TX_ID', 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_TYPE', 'TX_AMOUNT', 'TIMESTAMP', 'IS_FRAUD', 'ALERT_ID']
['ALERT_ID', 'ALERT_TYPE', 'IS_FRAUD', 'TX_ID', 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_TYPE', 'TX_AMOUNT', 'TIMESTAMP']


In [ ]:
# duplicates remove: har ALERT_ID ke liye sirf ek row rakhre (ALERT_TYPE toh same hi hoga)
alert_type_map = alerts[['ALERT_ID', 'ALERT_TYPE']].drop_duplicates()

print(alert_type_map.shape)
# ab yeh 391 hai (unique rings ki tarah), 1719 nahi

(391, 2)


In [ ]:
print(transactions.shape)
print(transactions['TX_ID'].duplicated().sum())

# Shape = (1323234, 9) original row count same hai, koi duplicate nahi bana
# Duplicated TX_ID = 0 — har transaction sirf ek baar hai

(1323234, 8)
0


In [ ]:
# transactions mein merge kara
transactions = transactions.merge(alert_type_map, on='ALERT_ID', how='left')
print(transactions.columns.tolist())  # ab ALERT_TYPE dikhna chahiye

['TX_ID', 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_TYPE', 'TX_AMOUNT', 'TIMESTAMP', 'IS_FRAUD', 'ALERT_ID', 'ALERT_TYPE']


In [ ]:
transactions[transactions['ALERT_TYPE'].notna()].head()

,TX_ID,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_TYPE,TX_AMOUNT,TIMESTAMP,IS_FRAUD,ALERT_ID,ALERT_TYPE
81,82,6976,9739,TRANSFER,4.85,0,True,193,fan_in
948,949,5776,2570,TRANSFER,10.27,0,True,377,cycle
6279,6280,9999,9530,TRANSFER,2.74,1,True,189,fan_in
7998,7999,1089,7352,TRANSFER,10.27,1,True,377,cycle
12974,12975,7025,9708,TRANSFER,3.53,2,True,130,fan_in


Humein transactions dataframe ko ek graph mein badalna hai:

1. Har account ek node (point) banega
2. Har transaction ek edge (arrow, sender → receiver) banega

In [ ]:
import networkx as nx

# we can't take DiGraph as
# normal DiGraph allows sirf EK edge between do same nodes.
# agar same sender receiver ke beech multiple transactions hue, toh purana edge overwrite ho jaata (jaise 1.3M transactions se sirf 68947 edges bane the).
# MultiDiGraph multiple edges allow karta hai same pair ke beech, isliye har transaction alag se preserve rehti hai.
G = nx.MultiDiGraph()

# nodes add (accounts)
for _, row in accounts.iterrows():
    G.add_node(row['ACCOUNT_ID'], is_fraud=row['IS_FRAUD'], behavior_id=row['TX_BEHAVIOR_ID'])

# edges list made (sender, receiver, amount) tuples ki
edges = list(zip(transactions['SENDER_ACCOUNT_ID'], transactions['RECEIVER_ACCOUNT_ID'], transactions['TX_AMOUNT']))

# edges add graph mein
G.add_weighted_edges_from(edges)

print(G.number_of_nodes())
print(G.number_of_edges())


#what we are doing
# example, 3 accounts hain — 101, 102, 103. Aur 2 transactions hue:
# Nodes (accounts se bane): 101, 102, 103 — teen points, chahe unki transaction ho ya na ho.
# edges:-
# 101 ne 102 ko ₹500 bheje
# 101 ne 102 ko phir se ₹300 bheje (dusri transaction)

10000
1323234


In [ ]:
import itertools

# poore graph pe try karne se pehle, sirf pehle 5 cycles nikaalo (taaki crash na ho, bahut bada graph hai)
cycles_sample = list(itertools.islice(nx.simple_cycles(G), 5))
print(cycles_sample)

#Cycle matlab: paisa ek account se nikal ke, kuch accounts se ho ke, wapas usi account tak aa jaye

# alerts.csv mein "cycle" label already diya hua tha, lekin real fraud detection mein aisa koi pehle se pata label nahi milega.
# isliye ab hum khud graph algorithm (simple_cycles) se cycles dhoond rahe hain, bina answer key dekhe.
# taaki baad mein check kar sakein ki humara khud ka detection alerts.csv ke "cycle" labels se match karta hai ya nahi.

#output:-
# 237 ne khud ko hi paisa bheja (237 → 237) -->self loops
# this is an edge case hai data mein jaha sender aur receiver same account hai.
# simple_cycles() function sabse chhote cycles pehle deta hai so length-1 (self loops) we got pehle.

[[237], [4009], [4765], [5718], [5742]]


In [ ]:
# check karo kitni transactions aisi hain jaha sender aur receiver same account hai
self_loops = transactions[transactions['SENDER_ACCOUNT_ID'] == transactions['RECEIVER_ACCOUNT_ID']]
print(len(self_loops))

181


In [ ]:
# self loops ko skip karke real cycles dhoondo (length > 1)
real_cycles = []
for cycle in itertools.islice(nx.simple_cycles(G), 1000):  # pehle 1000 cycles check karo
    if len(cycle)>1:  # sirf woh cycles rakho jisme 2+ accounts hain
        real_cycles.append(cycle)

print(len(real_cycles))  # kitne real cycles mile
print(real_cycles[:5])   # pehle 5 dikhao

989
[[3, 9556, 9960, 8378, 8751, 7340, 7807, 9991, 5140, 2746, 9147, 5236, 1668, 6165, 9872, 6670, 9861, 7046, 9993, 5141, 9950, 9982, 9862, 8210, 8463, 9994, 9983, 8772, 9114, 709, 9423, 9951, 8453, 6098, 5929, 8211, 8333, 8970, 5442, 7808, 9223, 8002, 4384, 9510, 8099, 7156, 5284, 9884, 5065, 9774, 7761, 7287, 5152, 4052, 6362, 9034, 9585, 9102, 9895, 5142, 8992, 9972, 9980, 8376, 9531, 9887, 7000, 3259, 4606, 9620, 7187, 9554, 3010, 9125, 8344, 7341, 6719, 9663, 44, 6773, 6055, 2426, 1117, 8056, 8113, 7131, 9499, 9883, 7960, 9012, 8783, 8176, 9992, 8015, 5483, 3218, 8336, 9961, 3082, 5959, 88, 7479, 1027, 9399, 9389, 9762, 9685, 9003, 6274, 5728, 9684, 2190, 8909, 6232, 6322, 8387, 4691, 8123, 8110, 5283, 5254, 9496, 8079, 8982, 9477, 9267, 5790, 5185, 298, 6221, 9965, 9226, 5481, 5673, 4519, 2765, 8809, 9971, 9765, 8057, 8311, 9045, 6928, 6562, 9975, 7285, 6991, 2757, 7693, 8388, 7686, 8321, 9795, 9268, 9894, 9388, 9850, 7045, 7010, 3290, 8167, 9386, 5880, 9784, 8199, 4293, 4040, 9

In [ ]:
#  sirf cycle detection ke liye ek simple DiGraph banaya so duplicate transactions collapse ho jayengi
# bas yeh dikhega ki A se B connection hai ya nahi amount/count yahan nahi chahiye abhi

G_simple = nx.DiGraph()
G_simple.add_edges_from(zip(transactions['SENDER_ACCOUNT_ID'], transactions['RECEIVER_ACCOUNT_ID']))

print(G_simple.number_of_nodes(), G_simple.number_of_edges())

9999 68947


In [34]:
# Cycle sirf "strongly connected components" (SCC) ke andar hi ban sakta hai
# matlab jaha A se B aur B se wapas A tak rasta ho.
# Agar koi group aisa nahi hai, uspe cycle check karna waste hai.
# so looking for thesebgroups
sccs = [c for c in nx.strongly_connected_components(G_simple) if len(c) > 1]

print(len(sccs))                          # kitne aise groups hain
print(max(len(c) for c in sccs))          # sabse bada group kitna bada hai

21
9849


In [ ]:
# Findings:
#   simple_cycles() poore graph pe / MultiDiGraph pe bahut slow tha (crash ho raha tha)
#   kyunki MultiDiGraph mein parallel edges (same sender receiver, multiple tx) ko
#   alag-alag cycles ki tarah count kar raha tha isliye simple DiGraph banaya (sirf
#   connection dikhta hai, count/amount nahi cycle shape dhoondne ke liye kaafi hai)
#
#   simple DiGraph pe bhi slow raha kyunki humara graph bahut DENSE hai:
#   21 strongly connected components mile, lekin sabse bada component 9849 accounts ka hai
#   (10000 mein se) matlab almost SAARA graph ek hi bade interconnected "blob" mein hai
#
#   itni density mein simple_cycles() lakhon random chhote cycles dhoond lega jo asal
#   fraud nahi hain, bas itni connectivity ki wajah se koi bhi kahin se ghoom ke wapas
#   aa sakta hai so naive/generic cycle search yahan practically kaam nahi karega
#
#  Next step: poore graph mein blind search karne ke bajaye, sirf fraud flagged
#  accounts (IS_FRAUD=True) ke chhote subgraph pe focus karke cycles dhoondhe